# Module 04 — AI Workbook Companion (Python on GPUs / HEP)

A lightweight companion to
[Session3_Python-GPU_HEP.ipynb](../Session3_Python-GPU_HEP.ipynb) and the
lesson-4 notebooks. It does **not** replace them — it adds the supervision layer
from [Module 11](../../11/README.md) to Module 04's topic: **array-oriented,
GPU-style physics analysis in Python** (CuPy / Numba).

New here? Read [Module 11's README](../../11/README.md) first — it explains the
5-phase loop, what a "CPU reference" is, and the known ways AI gets parallel code
wrong. The three failures translate directly from CUDA to Python-on-GPU: races
(scatter/`+=` without atomics), silent dtype/precision changes, and reductions
over the wrong axis.

## The loop, applied to a physics histogram

Your task: histogram a per-event observable (e.g. an energy or a transverse
momentum) across millions of events, the way a GPU analysis would — with a
vectorised scatter into bins. You may have an AI write it. Your job is everything
around it.

1. **SPECIFY** — Contract: input array shape and dtype, bin edges, what "correct"
   means (every event counted exactly once), and the metric (events/sec, or just
   correctness for this exercise).
2. **GENERATE** — Ask the AI for the vectorised histogram. Keep it in its own file.
3. **PREDICT** — *Before running:* the classic risk is a **scatter race** —
   `hist[idx] += 1` with repeated indices does **not** accumulate. Do you see it?
   (On a GPU this is the "no atomicAdd" bug; in NumPy it is the identical trap.)
4. **VERIFY** — Use [verify_histogram.py](./verify_histogram.py): it has a CPU
   reference (`np.bincount`) and a PASS/FAIL gate over the whole histogram.
5. **DIAGNOSE** — Explain why the buggy version undercounts, and what the GPU
   analog (`cuda.atomic.add`) would be.

## The adversarial exercise

[adversarial_histogram_buggy.py](./adversarial_histogram_buggy.py) is a
histogram "an AI wrote for you." It runs without error and the shape looks right,
but it **undercounts every bin** because `hist[idx] += 1` on a NumPy array with
repeated indices applies each duplicate only once — exactly the CPU mirror of a
GPU histogram with no atomic add.

Your job:

1. Read it and predict the failure *before* running.
2. Run it and compare its total count to the number of events — it will be short.
3. State the exact bug and the fix (`np.add.at` / `np.bincount`; on GPU,
   `cuda.atomic.add`).
4. Prove the fix bin-by-bin against the CPU reference.

Could an AI catch this? Sometimes — but "it ran and the plot looks like a
histogram" passes a casual review while the counts are silently wrong. A
bin-by-bin check against a reference catches it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why does `hist[idx] += 1` lose counts when `idx` has duplicates? How is that
  the same bug as a GPU histogram without `atomicAdd`?
- If you had only checked the histogram's *shape* and that it "looked
  bell-shaped," would you have caught the undercount? What check actually catches it?
- Where would `float32` vs `float64` change your bin assignment near an edge, and
  how would you test for it?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_histogram_buggy.py](./adversarial_histogram_buggy.py). Read it first and predict what will happen before you run this cell.

In [ ]:
!python adversarial_histogram_buggy.py

## Step 2 - Run your verification harness

[verify_histogram.py](./verify_histogram.py) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the function under test. Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!python verify_histogram.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.